# Chapter 3 — Speed and Accuracy: Implicit and Weighted Finite-Difference Schemes

This chapter adapts Chapter 3, *"Increasing the speed and accuracy of
computations"*, of Michael Honeychurch's **Simulating Electrochemical
Reactions in Mathematica** (SERM). The explicit scheme of Chapter 2 is simple
but conditionally stable: it requires the model diffusion coefficient
$D_M = D\,\Delta t/\Delta x^2 \le \tfrac12$, and it is only first order in
time. Here we develop the **fully implicit** (backward-Euler) scheme, the
**Crank–Nicolson** weighted scheme, and the **Douglas** higher-order variant,
all built on the tridiagonal solver `serm.tridiagonal`. We then show how an
**expanding spatial grid** concentrates resolution near the electrode where the
concentration gradient is steepest.

The deliverable: a vectorised numpy/scipy implementation of the $\theta$-weighted
family of schemes, validated three ways — (i) the implicit method is *stable for
$D_M > \tfrac12$* where the explicit method blew up in Chapter 2, (ii) Crank–
Nicolson is *second order in time* while backward Euler is *first order*, and
(iii) the simulated current matches the analytical **Cottrell** transient.

## The $\theta$-weighted family

Fick's second law in dimensionless form (Chapter 2) is
$\partial c/\partial \tau = \partial^2 c/\partial \chi^2$. Discretising the
right-hand side with the standard three-point second difference but evaluating
it as a **weighted average** of the new time level $k+1$ and the old level $k$
gives the one-parameter family

$$
\frac{c_j^{k+1}-c_j^{k}}{\Delta\tau}
= \frac{1}{\Delta\chi^2}\Big[\theta\,\delta^2 c_j^{k+1}
        + (1-\theta)\,\delta^2 c_j^{k}\Big],
\qquad
\delta^2 c_j \equiv c_{j-1}-2c_j+c_{j+1}.
$$

With $D_M = \Delta\tau/\Delta\chi^2$ and collecting the unknown $k+1$ terms on
the left:

$$
-\theta D_M\,c_{j-1}^{k+1}
+(1+2\theta D_M)\,c_j^{k+1}
-\theta D_M\,c_{j+1}^{k+1}
=
(1-\theta)D_M\,c_{j-1}^{k}
+\big[1-2(1-\theta)D_M\big]c_j^{k}
+(1-\theta)D_M\,c_{j+1}^{k}.
$$

Three members of this family interest us:

| $\theta$ | scheme | stability | temporal order |
|---|---|---|---|
| $0$ | explicit (forward Euler) | $D_M \le \tfrac12$ | 1 |
| $1$ | fully implicit (backward Euler) | unconditional | 1 |
| $\tfrac12$ | Crank–Nicolson | unconditional | 2 |

For $\theta>0$ the left-hand side couples the three unknowns $c_{j-1}^{k+1},
c_j^{k+1}, c_{j+1}^{k+1}$, so each time step is a **tridiagonal linear solve**
$A\mathbf{u}=\mathbf{b}$ rather than an explicit update. That is the price of
unconditional stability — and it is a price worth paying, because we can now
take large $D_M$ (large time steps or coarse space) without the solution
exploding.

### The implicit and Crank–Nicolson matrices

Honeychurch writes the matrices through the helper `makeDiagonals` /
`makeCrankDiagonals`. For the **fully implicit** scheme ($\theta=1$) the
interior system has sub/super-diagonal $-D_M$ and main diagonal $1+2D_M$, with
right-hand side simply the previous time column $c_j^{k}$ (`implicitSolve2` in
the source). For **Crank–Nicolson** ($\theta=\tfrac12$) Honeychurch multiplies
through by $2/D_M$ so the diagonals become $-1$ (off) and $2+2/D_M$ (main), and
the right-hand side is the correlation
$c_{j-1}^{k}+\big(\tfrac{2}{D_M}-2\big)c_j^{k}+c_{j+1}^{k}$ (`crankSolve2`,
formed with `ListCorrelate[{1, -2+2/D_M, 1}, ...]`). The boundary values
($c=0$ at the electrode, $c=1$ in the bulk) are moved to the right-hand side:
the last interior equation picks up the bulk term, the first picks up the
surface term (here zero).

We implement the whole family with one vectorised function and select the
scheme by $\theta$.

In [1]:
import os, sys
# Make the project package importable when the notebook runs from notebooks/.
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import numpy as np
import matplotlib.pyplot as plt

import serm
from serm.grids import make_grid, space_points, dx_dimensionless
from serm.tridiagonal import tridiag_solve_banded
from serm.echem import cottrell_current as cottrell_dimensional  # not used directly; see serm.cottrell_current

np.set_printoptions(precision=4, suppress=True)

In [2]:
def theta_solve(D_M, n, theta, m=None):
    """Solve the dimensionless Cottrell problem with a theta-weighted FD scheme.

    Implements the one-parameter family
        -theta*D_M c[j-1]^{k+1} + (1+2 theta D_M) c[j]^{k+1} - theta*D_M c[j+1]^{k+1}
        = (1-theta)D_M c[j-1]^k + (1-2(1-theta)D_M) c[j]^k + (1-theta)D_M c[j+1]^k
    for the potential step to the diffusion limit: surface c=0, bulk c=1,
    initial c=1.  theta=1 is fully implicit (backward Euler); theta=0.5 is
    Crank-Nicolson; theta=0 reduces to the explicit scheme of Chapter 2.

    Parameters
    ----------
    D_M : float
        Model diffusion coefficient D dt / dx^2. May exceed 0.5 for theta>0.
    n : int
        Number of time grid points.
    theta : float
        Weighting in [0, 1].
    m : int, optional
        Number of space points; defaults to serm.grids.space_points(D_M, n).

    Returns
    -------
    ndarray, shape (m, n)
        Concentration grid (space x time).
    """
    if m is None:
        m = space_points(D_M, n)
    c = make_grid(m, n)
    k_unk = m - 2                      # interior unknowns, rows 1 .. m-2
    a = theta * D_M                    # implicit (LHS) coupling
    b_w = (1.0 - theta) * D_M          # explicit (RHS) coupling

    sub = np.full(k_unk - 1, -a)       # x : sub-diagonal of A
    sup = np.full(k_unk - 1, -a)       # z : super-diagonal of A
    diag = np.full(k_unk, 1.0 + 2.0 * a)   # y : main diagonal of A

    for k in range(1, n):
        prev = c[:, k - 1]
        # Explicit (RHS) part: (1-theta)D_M * delta^2 + identity on interior nodes.
        rhs = (b_w * prev[0:m - 2]
               + (1.0 - 2.0 * b_w) * prev[1:m - 1]
               + b_w * prev[2:m]).astype(float)
        # Boundary contributions to the implicit operator move to the RHS:
        #   first interior eq gains a * c_surface (= 0); last gains a * c_bulk (= 1).
        rhs[-1] += a * 1.0
        if theta == 0.0:
            c[1:m - 1, k] = rhs        # explicit: A is the identity
        else:
            c[1:m - 1, k] = tridiag_solve_banded(sub, diag, sup, rhs)
    return c

We use `tridiag_solve_banded` (LAPACK's pivoting banded solver) rather than
the bare Thomas algorithm: for $\theta>0$ the matrix $1+2\theta D_M$ on the
diagonal is strongly diagonally dominant, so both work, but the banded solver is
the recommended production path (see `serm.tridiagonal`).

## Concentration profiles and the current transient

We run the fully implicit and Crank–Nicolson schemes at the same $D_M$ and
compare their concentration profiles and the current they predict. The
dimensionless current is the surface flux, computed by `serm.electrode_current`
with the accurate three-point one-sided derivative; the analytical reference is
the Cottrell form $i(\tau)=1/\sqrt{\pi\tau}$ (`serm.cottrell_current`).

In [3]:
D_M = 1.5          # NOTE: > 0.5 -- the explicit scheme of Ch.2 is unstable here
n   = 400
m   = space_points(D_M, n)
dx  = dx_dimensionless(D_M, n)
print(f"D_M = {D_M}  (> 0.5),  n = {n},  m = {m},  dx = {dx:.4f}")

c_impl  = theta_solve(D_M, n, theta=1.0)
c_crank = theta_solve(D_M, n, theta=0.5)
print("max |c| implicit :", np.nanmax(np.abs(c_impl)))
print("max |c| crank    :", np.nanmax(np.abs(c_crank)))

D_M = 1.5  (> 0.5),  n = 400,  m = 148,  dx = 0.0409
max |c| implicit : 1.0
max |c| crank    : 1.0


In [4]:
x = np.arange(m) * dx
taus = [0.02, 0.1, 0.3, 0.6, 1.0]
tau_axis = np.arange(n) / (n - 1)
cols = [int(round(t * (n - 1))) for t in taus]

fig, ax = plt.subplots(figsize=(5.8, 4.2))
for t, kcol in zip(taus, cols):
    ax.plot(x, c_crank[:, kcol], lw=1.2, label=fr"$\tau={t:.2f}$")
ax.set_xlabel(r"dimensionless distance $\chi$")
ax.set_ylabel(r"$c(\chi,\tau)$")
ax.set_title("Crank-Nicolson concentration profiles ($D_M=1.5$)")
ax.set_xlim(0, 6); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

/tmp/ipykernel_1422419/3963997978.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [5]:
i_impl  = serm.electrode_current(c_impl,  D_M)
i_crank = serm.electrode_current(c_crank, D_M)
i_cot   = serm.cottrell_current(n)
tau     = np.arange(n) / (n - 1)

fig, ax = plt.subplots(figsize=(5.8, 4.2))
ax.plot(tau, i_cot,  "k--", lw=1.2, label="Cottrell (analytical)")
ax.plot(tau, i_impl,  lw=1.0, label="fully implicit")
ax.plot(tau, i_crank, lw=1.0, label="Crank-Nicolson")
ax.set_xlabel(r"dimensionless time $\tau$")
ax.set_ylabel("dimensionless current")
ax.set_ylim(0, 6); ax.set_xlim(0, 1)
ax.set_title(r"Current transient at $D_M=1.5$")
ax.legend(); plt.tight_layout(); plt.show()

/tmp/ipykernel_1422419/3782219760.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.legend(); plt.tight_layout(); plt.show()


## The Douglas scheme — a near-free accuracy boost

Letting $\theta = \tfrac12 - \tfrac{1}{12 D_M}$ yields the **Douglas** scheme,
which is no harder to solve than Crank–Nicolson (still tridiagonal) but is
$\mathcal{O}(\Delta\chi^4)$ in space rather than $\mathcal{O}(\Delta\chi^2)$.
Substituting that $\theta$ into the general stencil, Honeychurch's
`makeDouglasDiagonals` / `douglasSolve` give the integer-friendly form with
off-diagonals $1-6D_M$, main diagonal $10+12D_M$, and right-hand-side kernel
$\{1+6D_M,\,10-12D_M,\,1+6D_M\}$. We implement it directly.

In [6]:
def douglas_solve(D_M, n, m=None):
    """Douglas scheme (theta = 1/2 - 1/(12 D_M)), O(dx^4) in space.

    Port of makeDouglasDiagonals / douglasSolve from chapter3.nb: off-diagonals
    1-6 D_M, main diagonal 10+12 D_M, RHS correlation kernel
    {1+6 D_M, 10-12 D_M, 1+6 D_M}.
    """
    if m is None:
        m = space_points(D_M, n)
    c = make_grid(m, n)
    k_unk = m - 2
    off = 1.0 - 6.0 * D_M
    sub = np.full(k_unk - 1, off)
    sup = np.full(k_unk - 1, off)
    diag = np.full(k_unk, 10.0 + 12.0 * D_M)
    kp, km = 1.0 + 6.0 * D_M, 10.0 - 12.0 * D_M
    for k in range(1, n):
        prev = c[:, k - 1]
        rhs = (kp * prev[0:m - 2] + km * prev[1:m - 1] + kp * prev[2:m]).astype(float)
        rhs[-1] -= off * 1.0          # bulk BC term (=-(1-6 D_M)*c_bulk)
        c[1:m - 1, k] = tridiag_solve_banded(sub, diag, sup, rhs)
    return c

c_doug = douglas_solve(D_M, n)
i_doug = serm.electrode_current(c_doug, D_M)
print("max |c| douglas:", np.nanmax(np.abs(c_doug)))

max |c| douglas: 1.0


In [7]:
# Accuracy comparison: simulated/analytical at every 40th step (skip the
# discontinuity at tau -> 0, which all schemes resolve poorly).
mask = (tau > 0.05)
idx = np.arange(0, n, 40)
idx = idx[(idx > 0) & (tau[idx] > 0.05)]
print(f"{'tau':>7} {'implicit':>10} {'crank':>10} {'douglas':>10}   (sim / Cottrell)")
for j in idx:
    print(f"{tau[j]:7.3f} {i_impl[j]/i_cot[j]:10.5f} "
          f"{i_crank[j]/i_cot[j]:10.5f} {i_doug[j]/i_cot[j]:10.5f}")

    tau   implicit      crank    douglas   (sim / Cottrell)
  0.100    1.01276    1.00944    1.00907
  0.201    1.00631    1.00470    1.00453
  0.301    1.00420    1.00313    1.00301
  0.401    1.00314    1.00235    1.00226
  0.501    1.00251    1.00188    1.00181
  0.602    1.00209    1.00156    1.00151
  0.702    1.00179    1.00134    1.00129
  0.802    1.00157    1.00117    1.00113
  0.902    1.00139    1.00104    1.00100


## Expanding the spatial grid

The largest concentration gradients sit at the electrode surface; far away the
profile barely moves. A uniform grid fine enough at the surface wastes points in
the bulk. The fix is an **expanding grid**: map the physical distance $\chi$ to a
new coordinate $y$ in which equal $\Delta y$ steps correspond to geometrically
growing $\Delta\chi$ steps. Honeychurch uses $\chi = (e^{y}-1)/a$ with expansion
factor $a$, so

$$
\frac{dy}{d\chi}=\frac{a}{1+a\chi}=a\,e^{-y},
\qquad
\frac{d^2c}{d\chi^2}
= a^2 e^{-2y}\Big(\frac{\partial^2 c}{\partial y^2}-\frac{\partial c}{\partial y}\Big).
$$

The diffusion coefficient becomes **position dependent**, $D_p = D_M\,e^{-2y_p}$,
largest at the surface ($y=0$) and shrinking into the bulk — exactly where we want
the resolution. The stability criterion for the explicit version is now the local
$D_p < \tfrac12$. We demonstrate the explicit expanding-grid solver from the
source (`variableExplicitSolve1`), with $\alpha=\Delta y/2$ carrying the
first-derivative (drift) term.

In [8]:
def expanding_explicit_solve(D_M, n, m, y_lim):
    """Explicit FD on an exponentially expanding grid chi=(e^y-1)/a.

    Port of variableExplicitSolve1 (chapter3.nb): position-dependent model
    diffusion coefficient D_p = D_M * exp(-2 (p-1)/(m-1) * y_lim), with the
    drift term carried by alpha = dy/2.  y_lim is the y-coordinate of the bulk
    boundary (y_max = ln(1 + 6 a) in the book's sizing).
    """
    c = make_grid(m, n)
    p = np.arange(1, m - 1)                 # 0-based interior node indices
    s = np.exp(-2.0 * p / (m - 1) * y_lim)  # exp(-2 y_p), y_p = p/(m-1)*y_lim
    dy = y_lim / (m - 1)
    alpha = dy / 2.0
    w_up = D_M * s * (1.0 + alpha)          # coefficient of c[p+1]
    w_md = 1.0 - 2.0 * D_M * s              # coefficient of c[p]
    w_dn = D_M * s * (1.0 - alpha)          # coefficient of c[p-1]
    for k in range(1, n):
        prev = c[:, k - 1]
        c[1:m - 1, k] = (w_dn * prev[0:m - 2]
                         + w_md * prev[1:m - 1]
                         + w_up * prev[2:m])
    return c, s

# Size the expanding grid: bulk at y_max = ln(1 + 6 a); a = expansion factor.
a_exp = 0.5
y_lim = np.log(1.0 + 6.0 * a_exp)
m_exp = 60
# Local stability needs max D_p = D_M < 0.5; pick D_M = 0.4 and a matching n.
D_M_exp = 0.4
n_exp = 400
c_exp, s_p = expanding_explicit_solve(D_M_exp, n_exp, m_exp, y_lim)
print(f"a = {a_exp},  y_max = {y_lim:.4f},  m = {m_exp},  max local D_p = {D_M_exp*s_p.max():.4f}")
print("max |c| (expanding explicit):", np.nanmax(np.abs(c_exp)))

a = 0.5,  y_max = 1.3863,  m = 60,  max local D_p = 0.3816
max |c| (expanding explicit): 1.0


In [9]:
# Map y-grid back to physical distance chi to show the point clustering.
p_all = np.arange(m_exp)
y_all = p_all / (m_exp - 1) * y_lim
chi_all = (np.exp(y_all) - 1.0) / a_exp

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.8))
axes[0].plot(chi_all, np.zeros_like(chi_all), "o", ms=3)
axes[0].set_xlabel(r"physical distance $\chi$"); axes[0].set_yticks([])
axes[0].set_title(f"Expanding grid point spacing (a={a_exp})")
axes[0].set_xlim(left=-0.1)

tau_e = np.arange(n_exp) / (n_exp - 1)
for t in [0.05, 0.2, 0.5, 1.0]:
    kc = int(round(t * (n_exp - 1)))
    axes[1].plot(chi_all, c_exp[:, kc], "-o", ms=2, lw=0.8, label=fr"$\tau={t}$")
axes[1].set_xlabel(r"physical distance $\chi$"); axes[1].set_ylabel(r"$c$")
axes[1].set_title("Profiles on the expanding grid"); axes[1].legend(fontsize=8)
axes[1].set_xlim(0, 6)
plt.tight_layout(); plt.show()

/tmp/ipykernel_1422419/2263111629.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## Validation

We validate three independent claims, each with an `assert`.

**(1) Unconditional stability.** Chapter 2's explicit scheme requires
$D_M\le\tfrac12$ and overshoots wildly above it. We run the implicit and
Crank–Nicolson schemes at $D_M=1.5$ and $D_M=5$ and confirm the concentration
stays within the physical band $[0,1]$.

**(2) Temporal order: backward Euler is $\mathcal{O}(\Delta\tau)$, Crank–
Nicolson is $\mathcal{O}(\Delta\tau^2)$.** A clean order check must isolate the
*time*-discretisation error from the *space* error. We do this with a
**method-of-lines reference**: discretise space only (the same three-point
Laplacian), which turns the PDE into a linear ODE system
$\dot{\mathbf c}=L\mathbf c+\mathbf g$ whose solution is *exact in time*,
$\mathbf c(\tau)=\mathbf c_s+e^{L\tau}(\mathbf c_0-\mathbf c_s)$ via
`scipy.linalg.expm`. Stepping the same semidiscrete system with the
$\theta$-scheme and refining $\Delta\tau$, the error against that reference is
purely temporal, so its slope reveals the scheme's temporal order. (This is the
convergence/self-consistency strategy of the authoring spec, used because no
closed form isolates the temporal truncation error alone.)

**(3) Cottrell agreement.** The simulated current must match the analytical
$i=1/\sqrt{\pi\tau}$ over a window away from the $\tau\to0$ discontinuity.

In [10]:
# (1) Stability for D_M > 1/2, where the explicit scheme of Chapter 2 diverges.
for D_test in [1.5, 5.0]:
    for theta, name in [(1.0, "implicit"), (0.5, "crank")]:
        c_t = theta_solve(D_test, 400, theta=theta)
        peak = float(np.nanmax(np.abs(c_t)))
        print(f"D_M={D_test:4.1f}  {name:8s}: max|c| = {peak:.6f}")
        assert peak <= 1.0 + 1e-6, (D_test, name, peak)
print("PASS: implicit and Crank-Nicolson stay bounded for D_M = 1.5 and 5.0 "
      "(explicit was unstable above 0.5).")

D_M= 1.5  implicit: max|c| = 1.000000
D_M= 1.5  crank   : max|c| = 1.000000
D_M= 5.0  implicit: max|c| = 1.000000


D_M= 5.0  crank   : max|c| = 1.000000
PASS: implicit and Crank-Nicolson stay bounded for D_M = 1.5 and 5.0 (explicit was unstable above 0.5).


In [11]:
# (2) Temporal order via a method-of-lines (matrix-exponential) reference.
from scipy.linalg import expm

def _space_operator(m, dx):
    """Interior Laplacian L (k x k) and boundary vector g for dc/dt = L c + g."""
    k = m - 2
    L = (np.diag(np.full(k, -2.0))
         + np.diag(np.ones(k - 1), 1)
         + np.diag(np.ones(k - 1), -1)) / dx**2
    g = np.zeros(k); g[-1] = 1.0 / dx**2     # bulk BC c=1; surface BC c=0
    return L, g

def _theta_step(c, dt, L, g, theta):
    """One theta-weighted step of the semidiscrete system (dense)."""
    k = len(c); I = np.eye(k)
    A = I - theta * dt * L
    rhs = (I + (1.0 - theta) * dt * L) @ c + dt * g
    return np.linalg.solve(A, rhs)

m_v, dx_v, T_v = 81, 0.05, 0.5
L, g = _space_operator(m_v, dx_v)
c0 = np.ones(m_v - 2)
c_s = np.linalg.solve(L, -g)                 # steady state
c_ref = c_s + expm(L * T_v) @ (c0 - c_s)     # exact-in-time reference

orders = {}
for theta, name in [(1.0, "backward Euler"), (0.5, "Crank-Nicolson")]:
    errs, dts = [], []
    for N in [40, 80, 160, 320, 640]:
        dt = T_v / N
        c = c0.copy()
        for _ in range(N):
            c = _theta_step(c, dt, L, g, theta)
        errs.append(float(np.max(np.abs(c - c_ref)))); dts.append(dt)
    errs, dts = np.array(errs), np.array(dts)
    p = np.polyfit(np.log(dts), np.log(errs), 1)[0]   # slope = temporal order
    orders[name] = p
    print(f"{name:16s}: errors {errs}  ->  fitted order = {p:.3f}")

assert abs(orders["backward Euler"] - 1.0) < 0.1, orders
assert abs(orders["Crank-Nicolson"] - 2.0) < 0.1, orders
print("PASS: backward Euler is first order in time (~1.0), "
      "Crank-Nicolson is second order (~2.0).")

backward Euler  : errors [0.0034 0.0017 0.0009 0.0004 0.0002]  ->  fitted order = 1.000


Crank-Nicolson  : errors [0. 0. 0. 0. 0.]  ->  fitted order = 2.000
PASS: backward Euler is first order in time (~1.0), Crank-Nicolson is second order (~2.0).


In [12]:
# (3) Cottrell agreement for all three production schemes at D_M = 1.5.
tau = np.arange(n) / (n - 1)
win = (tau >= 0.1) & (tau <= 0.9)          # exclude the tau->0 singularity
i_cot = serm.cottrell_current(n)
for sim, name, tol in [(i_impl, "implicit", 1e-2),
                       (i_crank, "crank", 1e-2),
                       (i_doug, "douglas", 1e-2)]:
    rel = np.abs(sim[win] - i_cot[win]) / i_cot[win]
    mean_rel = float(np.nanmean(rel))
    print(f"{name:9s}: mean rel err (0.1<=tau<=0.9) = {mean_rel:.4e}")
    assert mean_rel < tol, (name, mean_rel)
print("PASS: implicit, Crank-Nicolson and Douglas all match the Cottrell "
      "transient to better than 1%.")

implicit : mean rel err (0.1<=tau<=0.9) = 3.4796e-03
crank    : mean rel err (0.1<=tau<=0.9) = 2.5950e-03
douglas  : mean rel err (0.1<=tau<=0.9) = 2.4974e-03
PASS: implicit, Crank-Nicolson and Douglas all match the Cottrell transient to better than 1%.


## Summary

We extended the explicit scheme of Chapter 2 into the $\theta$-weighted family
and implemented it as a single vectorised tridiagonal solver:

- **Fully implicit / backward Euler** ($\theta=1$) and **Crank–Nicolson**
  ($\theta=\tfrac12$) are **unconditionally stable** — verified to stay within
  $[0,1]$ at $D_M=1.5$ and $D_M=5$, where the explicit scheme of Chapter 2
  diverged.
- **Crank–Nicolson is second order in time** while backward Euler is first
  order — confirmed by fitting the error against a method-of-lines reference
  (fitted slopes $\approx 2.0$ and $\approx 1.0$).
- The **Douglas** scheme ($\theta=\tfrac12-\tfrac1{12D_M}$) raises the spatial
  order to $\mathcal{O}(\Delta\chi^4)$ at the same tridiagonal cost.
- All three reproduce the analytical **Cottrell** current to better than 1%.
- An **expanding grid** ($\chi=(e^y-1)/a$) concentrates resolution at the
  electrode through a position-dependent local diffusion coefficient
  $D_p=D_M e^{-2y_p}$.

The tridiagonal solve at the heart of every implicit step is examined in
Chapter 4, where it is generalised to **block-tridiagonal** systems for coupled
homogeneous chemical reactions (Chapter 13). The unconditional stability earned
here is what makes the long simulations of AC voltammetry (Chapter 7), with tens
of thousands of time steps, tractable.